# OPPORTUNITY ENGINE — EXPLAINABILITY LAYER

In [10]:
import pandas as pd
import numpy as np

# 1. LOAD OPPORTUNITY ENGINE

engine = pd.read_csv("data/processed/opportunity_engine.csv")

In [11]:
# Clean text fields
engine["state"] = (
    engine["state"]
    .astype(str)
    .str.strip()
)

engine["industry"] = (
    engine["industry"]
    .astype(str)
    .str.strip()
)

# 2. SCORE COLUMNS

score_columns = {
    "Economic Strength": "economic_strength_score",
    "Industry Growth": "industry_growth_score",
    "Startup Momentum": "startup_momentum_score",
    "Underpenetration": "underpenetration_score",
    "Industry Presence": "industry_presence_score"
}


# 3. IDENTIFY TOP DRIVERS

def get_top_drivers(row, n=2):

    scores = {
        label: row[column]
        for label, column in score_columns.items()
        if pd.notna(row[column])
    }

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ", ".join(
        [x[0] for x in ranked[:n]]
    )


engine["top_drivers"] = engine.apply(
    get_top_drivers,
    axis=1
)


# 4. IDENTIFY WEAKNESSES

def get_weaknesses(row, n=2):

    scores = {
        label: row[column]
        for label, column in score_columns.items()
        if pd.notna(row[column])
    }

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1]
    )

    return ", ".join(
        [x[0] for x in ranked[:n]]
    )


engine["weakest_dimensions"] = engine.apply(
    get_weaknesses,
    axis=1
)


# 5. CREATE STRATEGY RECOMMENDATION

def strategy_recommendation(row):

    balanced = row["balanced_opportunity_score"]
    growth = row["growth_opportunity_score"]
    entry = row["market entry_opportunity_score"]

    scores = {
        "Balanced": balanced,
        "Growth": growth,
        "Market Entry": entry
    }

    best_strategy = max(
        scores,
        key=scores.get
    )

    return best_strategy


engine["best_strategy"] = engine.apply(
    strategy_recommendation,
    axis=1
)

In [12]:
# 6. CREATE BUSINESS INTERPRETATION

def generate_recommendation(row):

    score = row["balanced_opportunity_score"]
    drivers = row["top_drivers"]
    weaknesses = row["weakest_dimensions"]

    if score >= 65:

        recommendation = (
            "Strong market opportunity. "
            f"Key strengths: {drivers}. "
            f"Main consideration: {weaknesses}."
        )

    elif score >= 50:

        recommendation = (
            "Moderate market opportunity with "
            f"strength in {drivers}. "
            f"Potential constraints: {weaknesses}."
        )

    else:

        recommendation = (
            "Lower relative opportunity under the "
            f"balanced strategy. "
            f"Primary strengths: {drivers}; "
            f"constraints: {weaknesses}."
        )

    return recommendation


engine["business_recommendation"] = engine.apply(
    generate_recommendation,
    axis=1
)


# 7. SAVE ENHANCED ENGINE

engine.to_csv(
    "data/processed/opportunity_engine_final.csv",
    index=False
)

print("Enhanced Opportunity Engine saved.")
print("File: opportunity_engine_final.csv")

Enhanced Opportunity Engine saved.
File: opportunity_engine_final.csv


In [13]:
# 8. EXAMPLE STATE PROFILE

selected_industry = "Finance Technology"
selected_state = "Tamil Nadu"

profile = engine[
    (
        engine["industry"].str.lower()
        == selected_industry.lower()
    )
    &
    (
        engine["state"].str.lower()
        == selected_state.lower()
    )
]

print("\n" + "=" * 65)
print("STATE OPPORTUNITY PROFILE")
print("=" * 65)

if len(profile) == 0:

    print("No matching state-industry combination found.")

else:

    row = profile.iloc[0]

    print(f"\nState     : {row['state']}")
    print(f"Industry  : {row['industry']}")

    print(
        f"\nOverall Score : "
        f"{row['balanced_opportunity_score']:.2f}"
    )

    print(
        f"Overall Rank  : "
        f"{int(row['balanced_rank'])}"
    )

    print(
        f"Opportunity   : "
        f"{row['opportunity_tier']}"
    )

    print(
        f"\nBest Strategy : "
        f"{row['best_strategy']}"
    )

    print(
        f"\nTop Drivers   : "
        f"{row['top_drivers']}"
    )

    print(
        f"Weakest Areas : "
        f"{row['weakest_dimensions']}"
    )

    print(
        f"\nRecommendation:\n"
        f"{row['business_recommendation']}"
    )


STATE OPPORTUNITY PROFILE

State     : Tamil Nadu
Industry  : Finance Technology

Overall Score : 68.55
Overall Rank  : 1
Opportunity   : High Opportunity

Best Strategy : Market Entry

Top Drivers   : Industry Presence, Economic Strength
Weakest Areas : Startup Momentum, Underpenetration

Recommendation:
Strong market opportunity. Key strengths: Industry Presence, Economic Strength. Main consideration: Startup Momentum, Underpenetration.


In [14]:
# SCORECARD

scorecard_columns = [
    "economic_strength_score",
    "industry_growth_score",
    "startup_momentum_score",
    "underpenetration_score",
    "industry_presence_score"
]

profile_scores = profile[
    scorecard_columns
].T.reset_index()

profile_scores.columns = [
    "Dimension",
    "Score"
]

profile_scores["Score"] = (
    profile_scores["Score"].round(2)
)

print("\nSCORECARD")
print(profile_scores.to_string(index=False))


SCORECARD
              Dimension  Score
economic_strength_score  91.67
  industry_growth_score  75.00
 startup_momentum_score  25.00
 underpenetration_score  51.06
industry_presence_score 100.00


In [15]:
def get_market_profile(
    engine,
    state,
    industry
):

    result = engine[
        (engine["state"].str.lower() == state.lower())
        &
        (engine["industry"].str.lower() == industry.lower())
    ]

    if result.empty:
        return None

    return result.iloc[0]

In [16]:
profile = get_market_profile(
    engine,
    "Tamil Nadu",
    "Finance Technology"
)

print(profile)

year                                                                           2025
state                                                                    Tamil Nadu
industry                                                         Finance Technology
nsva                                                                      2896553.0
industry_startups                                                              60.0
industry_share                                                             0.034803
industry_cagr_3y                                                           2.326411
startup_cagr_3y                                                           -1.627675
economic_strength_score                                                       91.67
industry_growth_score                                                          75.0
startup_momentum_score                                                         25.0
underpenetration_score                                                      

In [17]:
def rank_markets(
    engine,
    industry,
    strategy="balanced"
):

    strategy_columns = {
        "balanced": "balanced_opportunity_score",
        "growth": "growth_opportunity_score",
        "market_entry": "market_entry_opportunity_score"
    }

    score_column = strategy_columns[strategy]

    result = engine[
        engine["industry"].str.lower()
        == industry.lower()
    ].copy()

    result = result.sort_values(
        score_column,
        ascending=False
    )

    result["rank"] = (
        result[score_column]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    return result

In [18]:
ranking = rank_markets(
    engine,
    "Finance Technology",
    strategy="balanced"
)

print(
    ranking[
        [
            "rank",
            "state",
            "balanced_opportunity_score",
            "top_drivers",
            "opportunity_tier"
        ]
    ].head(10)
)

     rank           state  balanced_opportunity_score  \
478     1      Tamil Nadu                       68.55   
425     2       Rajasthan                       64.16   
75      3           Bihar                       58.65   
218     4       Karnataka                       57.04   
127     5         Gujarat                       56.60   
18      6  Andhra Pradesh                       50.83   
322     7     Maharashtra                       50.19   
577     8     Uttarakhand                       49.39   
624     9     West Bengal                       47.08   
270    10  Madhya Pradesh                       45.61   

                              top_drivers      opportunity_tier  
478  Industry Presence, Economic Strength      High Opportunity  
425    Industry Growth, Industry Presence      High Opportunity  
75      Industry Growth, Startup Momentum      High Opportunity  
218  Industry Presence, Economic Strength  Moderate Opportunity  
127   Economic Strength, Startup Momentum 